# FinGuard AI — Data Audit & Preparation

**Objective.** Build a leakage-controlled, forward-looking corporate financial-distress classification pipeline. This notebook covers the panel audit, target construction, feature preparation, temporal evaluation design, and final out-of-time checks.

**Reading guide:** the analysis distinguishes contemporaneous classification from the harder one-year-ahead early-warning problem. Exploratory checks are retained only where they document an important methodological decision.


In [ ]:
# ============================================================
# FINANCIAL DISTRESS PROJECT
# STEP 1 — RAW DATASET LOAD & AUDIT
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

file_path = r"./data/distress_panel_FINAL.csv"

df = pd.read_csv(file_path, low_memory=False)

print("=" * 80)
print("FINANCIAL DISTRESS PROJECT — STEP 1")
print("=" * 80)

print("\nDATASET LOADED")
print("-" * 80)
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]:,}")

# ------------------------------------------------------------
# 2. COLUMN LIST
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COLUMN NAMES")
print("=" * 80)

for i, col in enumerate(df.columns, start=1):
    print(f"{i:3d}. {col}")

# ------------------------------------------------------------
# 3. DATA TYPES
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DATA TYPES")
print("=" * 80)

dtype_table = pd.DataFrame({
    "Column": df.columns,
    "Dtype": df.dtypes.astype(str).values,
    "Missing": df.isna().sum().values,
    "Missing_%": (df.isna().mean() * 100).round(2).values,
    "Unique": df.nunique(dropna=True).values
})

print(dtype_table.to_string(index=False))

# ------------------------------------------------------------
# 4. DUPLICATE ROWS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DUPLICATE CHECK")
print("=" * 80)

duplicate_rows = df.duplicated().sum()

print(f"Duplicate rows : {duplicate_rows:,}")

# ------------------------------------------------------------
# 5. SAMPLE RECORDS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FIRST 5 ROWS")
print("=" * 80)

print(df.head().to_string())

# ------------------------------------------------------------
# 6. NUMERIC SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("NUMERIC VARIABLE SUMMARY")
print("=" * 80)

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print(f"Numeric columns : {len(numeric_cols)}")

if len(numeric_cols) > 0:
    print(
        df[numeric_cols]
        .describe()
        .T
        .round(4)
        .to_string()
    )

# ------------------------------------------------------------
# 7. CATEGORICAL / OBJECT VARIABLES
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("NON-NUMERIC VARIABLES")
print("=" * 80)

non_numeric_cols = df.select_dtypes(
    exclude=[np.number]
).columns.tolist()

print(f"Non-numeric columns : {len(non_numeric_cols)}")

for col in non_numeric_cols:
    print(
        f"\n{col}"
        f"\n  dtype   : {df[col].dtype}"
        f"\n  unique  : {df[col].nunique(dropna=True):,}"
    )

    if df[col].nunique(dropna=True) <= 20:
        print(
            df[col]
            .value_counts(dropna=False)
            .head(20)
            .to_string()
        )

# ------------------------------------------------------------
# 8. POSSIBLE ID / TIME VARIABLES
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("POSSIBLE ID / TIME VARIABLES")
print("=" * 80)

keywords = [
    "id",
    "code",
    "company",
    "firm",
    "name",
    "year",
    "fy",
    "date",
    "period"
]

possible_id_time = [
    col for col in df.columns
    if any(keyword in col.lower() for keyword in keywords)
]

for col in possible_id_time:
    print(f"- {col}")

# ------------------------------------------------------------
# 9. FINAL OBJECT
# ------------------------------------------------------------

raw_df = df.copy()

print("\n" + "=" * 80)
print("STEP 1 COMPLETE")
print("=" * 80)

print(f"raw_df shape : {raw_df.shape}")

print("\nNext step:")
print("STEP 2 — identify firm ID, financial year, target variables,")
print("and construct the clean modelling panel.")

## 1. Data and panel structure

Establish the firm-year universe, observation structure, missingness, and basic data quality before modeling.


In [ ]:
# ============================================================
# FINANCIAL DISTRESS PROJECT
# STEP 2 — TARGET + PANEL STRUCTURE AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("FINANCIAL DISTRESS PROJECT — STEP 2")
print("=" * 80)

# ------------------------------------------------------------
# 1. WORKING COPY
# ------------------------------------------------------------

work_df = raw_df.copy()

# ------------------------------------------------------------
# 2. BASIC PANEL STRUCTURE
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PANEL STRUCTURE")
print("=" * 80)

print(f"Total observations : {len(work_df):,}")
print(f"Unique companies   : {work_df['Company'].nunique():,}")
print(f"FY range           : {work_df['FY'].min()} - {work_df['FY'].max()}")
print(f"Number of years    : {work_df['FY'].nunique()}")

# ------------------------------------------------------------
# 3. FIRM-YEAR UNIQUENESS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FIRM-YEAR UNIQUENESS")
print("=" * 80)

firm_year_duplicates = (
    work_df
    .duplicated(subset=["Company", "FY"])
    .sum()
)

print(f"Duplicate Company-FY observations : {firm_year_duplicates:,}")

if firm_year_duplicates == 0:
    print("✓ One observation per Company-FY.")
else:
    print("⚠ Duplicate Company-FY observations detected.")

# ------------------------------------------------------------
# 4. OBSERVATIONS BY YEAR
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("OBSERVATIONS BY FY")
print("=" * 80)

year_counts = (
    work_df
    .groupby("FY")
    .agg(
        Observations=("Company", "size"),
        Companies=("Company", "nunique"),
        Labeled=("DistressLabel", lambda x: x.notna().sum()),
        Missing_Label=("DistressLabel", lambda x: x.isna().sum())
    )
    .reset_index()
)

print(year_counts.to_string(index=False))

# ------------------------------------------------------------
# 5. LABEL DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DISTRESS LABEL DISTRIBUTION")
print("=" * 80)

label_counts = (
    work_df["DistressLabel"]
    .value_counts(dropna=False)
    .rename_axis("DistressLabel")
    .reset_index(name="Count")
)

label_counts["Percentage"] = (
    label_counts["Count"]
    / len(work_df)
    * 100
).round(2)

print(label_counts.to_string(index=False))

# ------------------------------------------------------------
# 6. LABEL DISTRIBUTION AMONG LABELED OBSERVATIONS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("LABEL DISTRIBUTION — LABELED OBSERVATIONS ONLY")
print("=" * 80)

labeled_df = work_df[
    work_df["DistressLabel"].notna()
].copy()

labeled_counts = (
    labeled_df["DistressLabel"]
    .value_counts()
    .rename_axis("DistressLabel")
    .reset_index(name="Count")
)

labeled_counts["Percentage"] = (
    labeled_counts["Count"]
    / len(labeled_df)
    * 100
).round(2)

print(labeled_counts.to_string(index=False))

# ------------------------------------------------------------
# 7. LABEL BY YEAR
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DISTRESS LABEL BY FY")
print("=" * 80)

label_by_year = pd.crosstab(
    work_df["FY"],
    work_df["DistressLabel"],
    dropna=False
)

print(label_by_year.to_string())

# ------------------------------------------------------------
# 8. LABEL PERCENTAGES BY YEAR
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DISTRESS LABEL PERCENTAGES BY FY")
print("=" * 80)

label_pct_by_year = (
    pd.crosstab(
        work_df["FY"],
        work_df["DistressLabel"],
        normalize="index"
    )
    * 100
)

print(label_pct_by_year.round(2).to_string())

# ------------------------------------------------------------
# 9. ICR ↔ DISTRESS LABEL CROSS-CHECK
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("ICR ↔ DISTRESS LABEL CROSS-CHECK")
print("=" * 80)

icr_label_df = work_df[
    work_df["ICR"].notna() &
    work_df["DistressLabel"].notna()
].copy()

print(f"Observations with both ICR and label : {len(icr_label_df):,}")

print("\nICR summary by DistressLabel:")

icr_summary = (
    icr_label_df
    .groupby("DistressLabel")["ICR"]
    .agg(
        Count="count",
        Min="min",
        Q25=lambda x: x.quantile(0.25),
        Median="median",
        Q75=lambda x: x.quantile(0.75),
        Max="max"
    )
)

print(icr_summary.round(4).to_string())

# ------------------------------------------------------------
# 10. TEST THE LIKELY ICR THRESHOLDS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TESTING ICR-BASED LABEL RULE")
print("=" * 80)

def expected_label_from_icr(icr):
    if pd.isna(icr):
        return np.nan

    if icr < 1:
        return "Distressed"

    elif icr <= 2:
        return "Vulnerable"

    else:
        return "Healthy"


icr_label_df["ExpectedLabel"] = (
    icr_label_df["ICR"]
    .apply(expected_label_from_icr)
)

icr_label_df["LabelMatches"] = (
    icr_label_df["DistressLabel"]
    == icr_label_df["ExpectedLabel"]
)

match_count = icr_label_df["LabelMatches"].sum()
comparison_count = len(icr_label_df)

match_pct = (
    match_count / comparison_count * 100
    if comparison_count > 0
    else np.nan
)

print(f"Matching observations : {match_count:,}")
print(f"Total compared        : {comparison_count:,}")
print(f"Match percentage      : {match_pct:.2f}%")

# ------------------------------------------------------------
# 11. MISMATCH TABLE
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("ICR / LABEL MISMATCH TABLE")
print("=" * 80)

mismatch_table = pd.crosstab(
    icr_label_df["ExpectedLabel"],
    icr_label_df["DistressLabel"]
)

print(mismatch_table.to_string())

# ------------------------------------------------------------
# 12. MISSING LABEL vs MISSING ICR
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MISSING LABEL vs MISSING ICR")
print("=" * 80)

missing_check = pd.DataFrame({
    "ICR_missing": work_df["ICR"].isna(),
    "Label_missing": work_df["DistressLabel"].isna()
})

missing_cross = pd.crosstab(
    missing_check["ICR_missing"],
    missing_check["Label_missing"]
)

print(missing_cross.to_string())

# ------------------------------------------------------------
# 13. UNIQUE VALUES OF LABEL
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DISTRESS LABEL VALUES")
print("=" * 80)

print(
    work_df["DistressLabel"]
    .dropna()
    .unique()
)

# ------------------------------------------------------------
# 14. FIRMS PER YEAR
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FIRM PANEL COVERAGE")
print("=" * 80)

firm_year_count = (
    work_df
    .groupby("Company")["FY"]
    .nunique()
)

print(
    firm_year_count
    .describe()
    .round(2)
    .to_string()
)

print(
    f"\nFirms appearing in only 1 year : "
    f"{(firm_year_count == 1).sum():,}"
)

print(
    f"Firms appearing in >= 3 years : "
    f"{(firm_year_count >= 3).sum():,}"
)

print(
    f"Firms appearing in >= 5 years : "
    f"{(firm_year_count >= 5).sum():,}"
)

# ------------------------------------------------------------
# 15. SAVE STEP 2 OBJECTS
# ------------------------------------------------------------

panel_audit = {
    "year_counts": year_counts,
    "label_counts": label_counts,
    "labeled_counts": labeled_counts,
    "label_by_year": label_by_year,
    "label_pct_by_year": label_pct_by_year,
    "icr_summary": icr_summary,
    "icr_label_df": icr_label_df,
    "mismatch_table": mismatch_table,
    "missing_cross": missing_cross,
    "firm_year_count": firm_year_count
}

print("\n" + "=" * 80)
print("STEP 2 COMPLETE")
print("=" * 80)

print("✓ Panel structure audited.")
print("✓ Target distribution audited.")
print("✓ ICR ↔ DistressLabel relationship tested.")
print("✓ Missing-label mechanism checked.")
print("✓ Firm-year structure checked.")

print("\nDO NOT MODEL YET.")
print("Next step will depend on the ICR/label consistency results.")

## 2. Target construction and early-warning framing

The distress label is constructed at the firm-year level. The early-warning extension shifts the target forward by one year so that predictors at time *t* are used to predict distress at *t+1*.


In [ ]:
# =============================================================================
# FINANCIAL DISTRESS PROJECT
# STEP 3 — DATA PREPARATION + ONE-YEAR-AHEAD TARGET
# =============================================================================
#
# Objective:
#   Use financial information in year t to predict the firm's distress state
#   in year t+1.
#
# Critical leakage rule:
#   ICR is EXCLUDED because DistressLabel is constructed almost entirely
#   from ICR.
#
# =============================================================================

import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

print("=" * 100)
print("FINANCIAL DISTRESS PROJECT — STEP 3")
print("ONE-YEAR-AHEAD PREDICTION DATASET")
print("=" * 100)


# =============================================================================
# STEP 3A — LOCATE / LOAD DATA
# =============================================================================

print("\n" + "=" * 100)
print("STEP 3A — DATA LOADING")
print("=" * 100)

# If raw_df already exists, use it.
# Otherwise load from the path you provided.

if "raw_df" in globals():
    df = raw_df.copy()
    print("✓ Using existing raw_df object.")
else:

    possible_paths = [
        r"./data/distress_panel_FINAL.csv",
        r"./data/distress_panel_FINAL.csv"
    ]

    loaded = False

    for path in possible_paths:
        try:
            df = pd.read_csv(path, low_memory=False)
            print(f"✓ Loaded dataset from:")
            print(path)
            loaded = True
            break
        except Exception:
            pass

    if not loaded:
        raise FileNotFoundError(
            "Could not load distress_panel_FINAL CSV. "
            "Either define raw_df or check the file path."
        )

print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns):,}")


# =============================================================================
# STEP 3B — STANDARDIZE BASIC COLUMNS
# =============================================================================

print("\n" + "=" * 100)
print("STEP 3B — BASIC STANDARDIZATION")
print("=" * 100)

df.columns = (
    df.columns
    .astype(str)
    .str.strip()
)

# Company as string
df["Company"] = df["Company"].astype(str).str.strip()

# FY numeric
df["FY"] = pd.to_numeric(
    df["FY"],
    errors="coerce"
)

# Sort panel
df = (
    df
    .sort_values(["Company", "FY"])
    .reset_index(drop=True)
)

print("✓ Columns standardized.")
print("✓ Panel sorted by Company and FY.")


# =============================================================================
# STEP 3C — CONVERT NUMERIC VARIABLES
# =============================================================================

print("\n" + "=" * 100)
print("STEP 3C — NUMERIC CONVERSION")
print("=" * 100)

identifier_cols = [
    "Company",
    "IndustryGroup",
    "DistressLabel"
]

numeric_cols = [
    c for c in df.columns
    if c not in identifier_cols
]

for col in numeric_cols:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

print(f"✓ Converted {len(numeric_cols)} columns to numeric where applicable.")


# =============================================================================
# STEP 3D — REPLACE INFINITY
# =============================================================================

print("\n" + "=" * 100)
print("STEP 3D — INFINITY CHECK")
print("=" * 100)

numeric_for_inf = df.select_dtypes(
    include=[np.number]
).columns

inf_counts = {}

for col in numeric_for_inf:

    count_inf = np.isinf(
        df[col].to_numpy(
            dtype=float,
            na_value=np.nan
        )
    ).sum()

    if count_inf > 0:
        inf_counts[col] = int(count_inf)

print("Infinite-value counts before cleaning:")

if len(inf_counts) == 0:
    print("None.")
else:
    for col, count in inf_counts.items():
        print(f"{col:<35} {count:>8,}")

# IMPORTANT:
# Replace +/- infinity with NaN.
# Do NOT impute yet.
df[numeric_for_inf] = df[numeric_for_inf].replace(
    [np.inf, -np.inf],
    np.nan
)

print("\n✓ +/- infinity replaced with NaN.")
print("✓ No arbitrary zero replacement performed.")


# =============================================================================
# STEP 3E — DEFINE TARGET LOGIC
# =============================================================================

print("\n" + "=" * 100)
print("STEP 3E — ONE-YEAR-AHEAD TARGET CONSTRUCTION")
print("=" * 100)

# The contemporaneous DistressLabel is based on ICR.
#
# We therefore create:
#
#   Target_t1 = DistressLabel at FY t+1
#
# Features remain measured at FY t.

df["Target_t1"] = (
    df
    .groupby("Company")["DistressLabel"]
    .shift(-1)
)

df["Target_FY"] = (
    df
    .groupby("Company")["FY"]
    .shift(-1)
)

# Check target alignment
target_alignment_ok = (
    df["Target_FY"]
    == df["FY"] + 1
)

print(
    "Correct FY(t) -> FY(t+1) alignment:",
    target_alignment_ok.dropna().mean()
)

# Last year of every firm naturally has no t+1 target.
print(
    f"Rows without next-year target: "
    f"{df['Target_t1'].isna().sum():,}"
)


# =============================================================================
# STEP 3F — TARGET DISTRIBUTION
# =============================================================================

print("\n" + "=" * 100)
print("ONE-YEAR-AHEAD TARGET DISTRIBUTION")
print("=" * 100)

target_counts = (
    df["Target_t1"]
    .value_counts(dropna=False)
    .rename_axis("Target_t1")
    .reset_index(name="Count")
)

target_counts["Percentage"] = (
    target_counts["Count"]
    / len(df)
    * 100
)

print(
    target_counts.to_string(
        index=False,
        formatters={
            "Percentage": "{:.2f}".format
        }
    )
)


# =============================================================================
# STEP 3G — EXPLICIT MODEL FEATURE SET
# =============================================================================

print("\n" + "=" * 100)
print("STEP 3G — MODEL FEATURE SET")
print("=" * 100)

# -------------------------------------------------------------------------
# CORE FINANCIAL DISTRESS VARIABLES
# -------------------------------------------------------------------------

core_features = [

    # Leverage
    "DebtEquityRatio",

    # Profitability
    "ROCE",

    # Cash-flow strength
    "CFO_to_TL",

    # Operating efficiency
    "AssetTurnover",

    # Liquidity
    "CurrentRatio",

    # Growth
    "SalesGrowth",

    # Size
    "LogTotalAssets",

    # Firm characteristics
    "FirmAge",

    # Governance / financial risk
    "PromoterPledgeRatio",

    # Additional financial structure
    "MarketToBook",
    "PromoterControlConcentration",
    "InstitutionalHolding_pct",

    # Investment / asset structure
    "CapexToAssets",
    "TangibleAssetRatio",

    # Earnings quality
    "Accruals_to_Assets",
    "CFO_to_NetIncome",

    # Liquidity / cash stress
    "CashBurnRunway",
    "CashConversionCycle",

    # Working-capital stress
    "DebtorDays",
    "InventoryDays",
    "CreditorDays",

    # Cash position
    "CashToAssets",

    # Governance disclosure
    "EquityRaised_dummy",
    "PledgeDisclosed",

    # Relative industry position
    "Industry_Median_Deviation",

    # Dynamic indicators
    "ROCE_3yr_trend",
    "SalesGrowth_3yr_trend",
    "Leverage_trend"
]


# Keep only columns actually present
model_features = [
    c for c in core_features
    if c in df.columns
]

missing_feature_candidates = [
    c for c in core_features
    if c not in df.columns
]

print(f"Requested candidate features : {len(core_features)}")
print(f"Available candidate features : {len(model_features)}")

if missing_feature_candidates:
    print("\nMissing candidate variables:")
    for c in missing_feature_candidates:
        print(" -", c)

print("\nCandidate feature list:")
for i, c in enumerate(model_features, 1):
    print(f"{i:>2}. {c}")


# =============================================================================
# STEP 3H — HARD EXCLUSIONS
# =============================================================================

print("\n" + "=" * 100)
print("STEP 3H — LEAKAGE CONTROL")
print("=" * 100)

# These MUST NOT enter the model.

forbidden_features = [
    "ICR",
    "DistressLabel",
    "Target_t1",
    "Target_FY"
]

model_features = [
    c for c in model_features
    if c not in forbidden_features
]

print("Explicitly excluded:")
for c in forbidden_features:
    print(" -", c)

print(
    f"\n✓ Final candidate feature count: "
    f"{len(model_features)}"
)


# =============================================================================
# STEP 3I — FEATURE AVAILABILITY AUDIT
# =============================================================================

print("\n" + "=" * 100)
print("STEP 3I — FEATURE AVAILABILITY")
print("=" * 100)

feature_audit_rows = []

for col in model_features:

    missing = df[col].isna().sum()

    feature_audit_rows.append({

        "Feature": col,

        "NonMissing": int(df[col].notna().sum()),

        "Missing": int(missing),

        "Missing_%": (
            missing / len(df) * 100
        ),

        "Unique": int(
            df[col].nunique(dropna=True)
        ),

        "Finite": int(
            np.isfinite(
                df[col].dropna()
            ).sum()
        )
    })


feature_audit = (
    pd.DataFrame(feature_audit_rows)
    .sort_values(
        "Missing_%",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    feature_audit.to_string(
        index=False,
        formatters={
            "Missing_%": "{:.2f}".format
        }
    )
)


# =============================================================================
# STEP 3J — DESCRIPTIVE STATISTICS
# =============================================================================

print("\n" + "=" * 100)
print("STEP 3J — FEATURE DISTRIBUTIONS")
print("=" * 100)

feature_summary = (
    df[model_features]
    .describe()
    .T
)

feature_summary["Missing"] = (
    df[model_features]
    .isna()
    .sum()
)

feature_summary["Missing_%"] = (
    feature_summary["Missing"]
    / len(df)
    * 100
)

print(
    feature_summary[
        [
            "count",
            "mean",
            "std",
            "min",
            "25%",
            "50%",
            "75%",
            "max",
            "Missing",
            "Missing_%"
        ]
    ]
    .round(4)
    .to_string()
)


# =============================================================================
# STEP 3K — TARGETED OUTLIER DIAGNOSTIC
# =============================================================================

print("\n" + "=" * 100)
print("STEP 3K — EXTREME VALUE DIAGNOSTIC")
print("=" * 100)

outlier_rows = []

for col in model_features:

    s = df[col].dropna()

    if len(s) == 0:
        continue

    q1 = s.quantile(0.01)
    q99 = s.quantile(0.99)

    extreme_low = (
        s < q1
    ).sum()

    extreme_high = (
        s > q99
    ).sum()

    outlier_rows.append({

        "Feature": col,

        "P01": q1,

        "Median": s.median(),

        "P99": q99,

        "Below_P01": int(extreme_low),

        "Above_P99": int(extreme_high),

        "Extreme_%": (
            (extreme_low + extreme_high)
            / len(s)
            * 100
        )
    })

outlier_summary = (
    pd.DataFrame(outlier_rows)
    .sort_values(
        "Extreme_%",
        ascending=False
    )
)

print(
    outlier_summary
    .round(4)
    .to_string(index=False)
)


# =============================================================================
# STEP 3L — BUILD MODELING DATASET
# =============================================================================

print("\n" + "=" * 100)
print("STEP 3L — BUILDING MODELING DATASET")
print("=" * 100)

# Require:
#   1. A valid t+1 target
#   2. At least one predictor
#
# DO NOT complete-case drop yet.
# Missing-value treatment will be done using TRAINING data only.

model_df = df[
    df["Target_t1"].notna()
].copy()

print(
    f"Rows with valid t+1 target: "
    f"{len(model_df):,}"
)

print(
    f"Companies represented: "
    f"{model_df['Company'].nunique():,}"
)

print(
    f"FY range of features: "
    f"{model_df['FY'].min()} - {model_df['FY'].max()}"
)

print(
    f"Target FY range: "
    f"{model_df['Target_FY'].min()} - "
    f"{model_df['Target_FY'].max()}"
)


# =============================================================================
# STEP 3M — TARGET YEAR CHECK
# =============================================================================

print("\n" + "=" * 100)
print("STEP 3M — TARGET YEAR CHECK")
print("=" * 100)

target_year_table = (
    model_df
    .groupby("FY")
    .agg(
        Feature_Year_Obs=("Company", "size"),
        Target_Year=("Target_FY", "first"),
        Target_Labeled=("Target_t1", "count")
    )
    .reset_index()
)

print(
    target_year_table
    .to_string(index=False)
)


# =============================================================================
# STEP 3N — TARGET DISTRIBUTION BY TARGET YEAR
# =============================================================================

print("\n" + "=" * 100)
print("STEP 3N — TARGET DISTRIBUTION BY TARGET YEAR")
print("=" * 100)

target_by_year = (
    pd.crosstab(
        model_df["Target_FY"],
        model_df["Target_t1"],
        normalize="index"
    )
    * 100
)

print(
    target_by_year
    .round(2)
    .to_string()
)


# =============================================================================
# STEP 3O — CLASS COUNTS
# =============================================================================

print("\n" + "=" * 100)
print("STEP 3O — CLASS COUNTS")
print("=" * 100)

final_target_counts = (
    model_df["Target_t1"]
    .value_counts()
)

print(
    final_target_counts
    .to_string()
)

print("\nClass proportions:")

print(
    (
        final_target_counts
        / final_target_counts.sum()
        * 100
    )
    .round(2)
    .to_string()
)


# =============================================================================
# STEP 3P — SAVE CORE OBJECTS
# =============================================================================

print("\n" + "=" * 100)
print("STEP 3P — SAVING OBJECTS")
print("=" * 100)

# Main objects for future cells
prepared_df = df.copy()

distress_model_df = model_df.copy()

candidate_features = model_features.copy()

print("✓ prepared_df created")
print("✓ distress_model_df created")
print("✓ candidate_features created")
print("✓ feature_audit created")
print("✓ feature_summary created")
print("✓ outlier_summary created")
print("✓ target_by_year created")


# =============================================================================
# STEP 3Q — FINAL SANITY CHECK
# =============================================================================

print("\n" + "=" * 100)
print("FINAL SANITY CHECK")
print("=" * 100)

print(
    f"Total panel observations : {len(df):,}"
)

print(
    f"Modeling observations     : {len(distress_model_df):,}"
)

print(
    f"Companies                 : "
    f"{distress_model_df['Company'].nunique():,}"
)

print(
    f"Candidate predictors      : "
    f"{len(candidate_features)}"
)

print(
    f"Target classes            : "
    f"{distress_model_df['Target_t1'].nunique()}"
)

print(
    "\nForbidden predictor check:"
)

leakage_present = [
    c for c in candidate_features
    if c in forbidden_features
]

if leakage_present:
    print("⚠ LEAKAGE VARIABLES PRESENT:", leakage_present)
else:
    print("✓ No ICR / label leakage variables in candidate features.")


print("\n" + "=" * 100)
print("STEP 3 COMPLETE")
print("=" * 100)

print(
    """
✓ One-year-ahead target constructed.
✓ ICR excluded from predictors.
✓ DistressLabel excluded from predictors.
✓ Infinite values converted to missing.
✓ Feature availability audited.
✓ Extreme-value diagnostic completed.
✓ Temporal target alignment verified.
✓ Modeling dataset created.

NEXT:
Temporal train/validation/test split + leakage-safe imputation +
winsorization/scaling + baseline multinomial logistic model.
"""
)

## 3. Feature audit and leakage control

The information-content of candidate predictors is checked before modeling. Interest Coverage Ratio (ICR) is excluded from the final feature set because the distress label is substantially constructed from ICR; retaining it would leak target information into the model.


In [ ]:
# =============================================================================
# FINANCIAL DISTRESS PROJECT
# STEP 3 — FEATURE AUDIT + LEAKAGE CONTROL + MODELING DATASET
# =============================================================================

import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

print("=" * 80)
print("FINANCIAL DISTRESS PROJECT — STEP 3")
print("=" * 80)

# -----------------------------------------------------------------------------
# 1. LOAD DATA
# -----------------------------------------------------------------------------

FILE_PATH = r"./data/distress_panel_FINAL.csv"

df = pd.read_csv(FILE_PATH, low_memory=False)

print("\nDATASET")
print("-" * 80)
print("Shape:", df.shape)
print("Columns:", len(df.columns))

# -----------------------------------------------------------------------------
# 2. BASIC COLUMN INFORMATION
# -----------------------------------------------------------------------------

print("\nCOLUMN LIST")
print("-" * 80)

for i, col in enumerate(df.columns):
    print(f"{i:3d} : {col}")

# -----------------------------------------------------------------------------
# 3. IDENTIFY KEY COLUMNS
# -----------------------------------------------------------------------------

company_col = "Company"
fy_col = "FY"
target_col = "DistressLabel"

required_cols = [company_col, fy_col, target_col]

missing_required = [
    c for c in required_cols
    if c not in df.columns
]

if missing_required:
    raise ValueError(
        f"Missing required columns: {missing_required}"
    )

print("\n✓ Required columns found.")

# -----------------------------------------------------------------------------
# 4. SORT PANEL
# -----------------------------------------------------------------------------

df[fy_col] = pd.to_numeric(df[fy_col], errors="coerce")

df = (
    df
    .sort_values([company_col, fy_col])
    .reset_index(drop=True)
)

# -----------------------------------------------------------------------------
# 5. TARGET CLEANING
# -----------------------------------------------------------------------------

print("\nTARGET DISTRIBUTION")
print("-" * 80)

print(df[target_col].value_counts(dropna=False))

# Keep only observations with known labels for supervised modeling
model_df = df[df[target_col].notna()].copy()

print("\nLabeled observations:", len(model_df))

# -----------------------------------------------------------------------------
# 6. TARGET ENCODING
# -----------------------------------------------------------------------------

label_map = {
    "Healthy": 0,
    "Vulnerable": 1,
    "Distressed": 2
}

model_df["Target"] = model_df[target_col].map(label_map)

if model_df["Target"].isna().any():
    unknown_labels = model_df.loc[
        model_df["Target"].isna(),
        target_col
    ].unique()

    raise ValueError(
        f"Unknown target labels found: {unknown_labels}"
    )

print("\nTARGET ENCODING")
print("-" * 80)
print(label_map)

print("\nEncoded target distribution:")
print(
    model_df["Target"]
    .value_counts()
    .sort_index()
)

# -----------------------------------------------------------------------------
# 7. IDENTIFY ICR
# -----------------------------------------------------------------------------

if "ICR" in model_df.columns:

    print("\n⚠ ICR DETECTED")
    print("-" * 80)

    print(
        "ICR is excluded from predictors because the Step 2 diagnostic "
        "showed that the DistressLabel is almost entirely determined by ICR."
    )

else:

    print("\nICR column not present.")

# -----------------------------------------------------------------------------
# 8. EXCLUDE LEAKAGE / IDENTIFIERS
# -----------------------------------------------------------------------------

leakage_cols = [
    target_col,
    "Target",
    "ICR"
]

identifier_cols = [
    company_col
]

# Any obvious label-construction columns should also be inspected.
possible_leakage = [
    c for c in model_df.columns
    if any(
        keyword in c.lower()
        for keyword in [
            "distress",
            "label",
            "default",
            "zscore",
            "altman",
            "interestcoverage"
        ]
    )
]

print("\nPOTENTIAL LEAKAGE COLUMNS")
print("-" * 80)

print(possible_leakage)

# Do NOT automatically remove every possible column here.
# We explicitly remove known target/leakage columns first.

excluded_cols = set(
    leakage_cols +
    identifier_cols
)

# -----------------------------------------------------------------------------
# 9. IDENTIFY NUMERIC FEATURES
# -----------------------------------------------------------------------------

numeric_cols = model_df.select_dtypes(
    include=[np.number]
).columns.tolist()

# Remove FY and Target from predictor candidates
candidate_features = [
    c for c in numeric_cols
    if c not in excluded_cols
    and c != fy_col
]

print("\nNUMERIC CANDIDATE FEATURES")
print("-" * 80)

for c in candidate_features:
    print(c)

print("\nNumber of candidate numeric features:", len(candidate_features))

# -----------------------------------------------------------------------------
# 10. MISSINGNESS AUDIT
# -----------------------------------------------------------------------------

missing_table = pd.DataFrame({
    "Feature": candidate_features,
    "Missing_Count": [
        model_df[c].isna().sum()
        for c in candidate_features
    ],
    "Missing_%": [
        model_df[c].isna().mean() * 100
        for c in candidate_features
    ]
})

missing_table = (
    missing_table
    .sort_values("Missing_%", ascending=False)
    .reset_index(drop=True)
)

print("\nMISSINGNESS AUDIT")
print("-" * 80)

print(missing_table.to_string(index=False))

# -----------------------------------------------------------------------------
# 11. REMOVE EXTREME-MISSING FEATURES
# -----------------------------------------------------------------------------

# Keep features with at least 60% observed values.
MAX_MISSING = 40.0

features_after_missing = missing_table.loc[
    missing_table["Missing_%"] <= MAX_MISSING,
    "Feature"
].tolist()

print("\nFEATURES RETAINED AFTER MISSINGNESS FILTER")
print("-" * 80)

for c in features_after_missing:
    print(c)

print(
    "\nFeatures retained:",
    len(features_after_missing)
)

# -----------------------------------------------------------------------------
# 12. CHECK CONSTANT FEATURES
# -----------------------------------------------------------------------------

constant_features = []

for c in features_after_missing:

    if model_df[c].nunique(dropna=True) <= 1:
        constant_features.append(c)

if constant_features:

    print("\nCONSTANT FEATURES REMOVED")
    print("-" * 80)

    print(constant_features)

features_after_constant = [
    c for c in features_after_missing
    if c not in constant_features
]

# -----------------------------------------------------------------------------
# 13. FINAL INITIAL FEATURE SET
# -----------------------------------------------------------------------------

feature_cols = features_after_constant

print("\n" + "=" * 80)
print("INITIAL MODEL FEATURE SET")
print("=" * 80)

for i, c in enumerate(feature_cols, 1):
    print(f"{i:2d}. {c}")

print("\nTotal features:", len(feature_cols))

# -----------------------------------------------------------------------------
# 14. FINAL MODELING DATASET
# -----------------------------------------------------------------------------

modeling_df = model_df[
    [company_col, fy_col, target_col, "Target"] + feature_cols
].copy()

print("\nMODELING DATASET")
print("-" * 80)
print("Shape:", modeling_df.shape)
print("Companies:", modeling_df[company_col].nunique())
print(
    "FY range:",
    modeling_df[fy_col].min(),
    "-",
    modeling_df[fy_col].max()
)

# -----------------------------------------------------------------------------
# 15. TARGET BY YEAR
# -----------------------------------------------------------------------------

year_target = pd.crosstab(
    modeling_df[fy_col],
    modeling_df[target_col],
    normalize="index"
) * 100

print("\nTARGET PERCENTAGES BY YEAR")
print("-" * 80)

print(
    year_target.round(2).to_string()
)

# -----------------------------------------------------------------------------
# 16. SAVE INTERMEDIATE DATASET
# -----------------------------------------------------------------------------

OUTPUT_PATH = r"./data/distress_modeling_base.csv"

modeling_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\n✓ Modeling dataset saved:")
print(OUTPUT_PATH)

# -----------------------------------------------------------------------------
# 17. FINAL SAFETY CHECK
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("LEAKAGE SAFETY CHECK")
print("=" * 80)

print("ICR in model features:", "ICR" in feature_cols)
print("Target in model features:", target_col in feature_cols)
print("Encoded Target in model features:", "Target" in feature_cols)
print("Company ID in model features:", company_col in feature_cols)

if "ICR" in feature_cols:
    raise ValueError(
        "CRITICAL: ICR is still present in model features."
    )

if target_col in feature_cols:
    raise ValueError(
        "CRITICAL: DistressLabel is still present in model features."
    )

print("\n✓ No direct ICR leakage.")
print("✓ No target leakage.")
print("✓ Dataset ready for temporal modeling.")

print("\n" + "=" * 80)
print("STEP 3 COMPLETE")
print("=" * 80)

## 4. Temporal split and feature engineering

Financial-risk models are evaluated using later periods rather than relying only on random splits. This better reflects the intended forward-looking use case.


In [ ]:
# =============================================================================
# FINANCIAL DISTRESS PROJECT
# STEP 4 — TEMPORAL SPLIT + FEATURE ENGINEERING AUDIT
# =============================================================================

import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

print("=" * 80)
print("FINANCIAL DISTRESS PROJECT — STEP 4")
print("=" * 80)

# =============================================================================
# 1. LOAD BASE MODELING DATA
# =============================================================================

BASE_PATH = r"./data/distress_modeling_base.csv"

modeling_df = pd.read_csv(
    BASE_PATH,
    low_memory=False
)

print("\nDATASET")
print("-" * 80)
print("Shape:", modeling_df.shape)

# =============================================================================
# 2. BASIC CHECKS
# =============================================================================

required_cols = [
    "Company",
    "FY",
    "DistressLabel",
    "Target"
]

missing_required = [
    c for c in required_cols
    if c not in modeling_df.columns
]

if missing_required:
    raise ValueError(
        f"Missing required columns: {missing_required}"
    )

modeling_df["FY"] = pd.to_numeric(
    modeling_df["FY"],
    errors="coerce"
)

modeling_df = (
    modeling_df
    .sort_values(["Company", "FY"])
    .reset_index(drop=True)
)

print("✓ Required columns found.")
print("✓ Data sorted by Company and FY.")

# =============================================================================
# 3. CHECK DUPLICATE FIRM-YEAR OBSERVATIONS
# =============================================================================

duplicates = modeling_df.duplicated(
    subset=["Company", "FY"]
).sum()

print("\nFIRM-YEAR DUPLICATES")
print("-" * 80)
print("Duplicate Company-FY rows:", duplicates)

if duplicates > 0:
    raise ValueError(
        "Duplicate Company-FY observations detected."
    )

print("✓ One observation per Company-FY.")

# =============================================================================
# 4. DEFINE TEMPORAL SPLIT
# =============================================================================
#
# 2012–2019 : TRAIN
# 2020–2021 : VALIDATION
# 2022–2023 : OUT-OF-TIME TEST
#
# This is deliberately chronological.
#
# =============================================================================

TRAIN_END = 2019
VALIDATION_START = 2020
VALIDATION_END = 2021
OOT_START = 2022
OOT_END = 2023

train_df = modeling_df[
    modeling_df["FY"] <= TRAIN_END
].copy()

validation_df = modeling_df[
    (modeling_df["FY"] >= VALIDATION_START)
    &
    (modeling_df["FY"] <= VALIDATION_END)
].copy()

oot_df = modeling_df[
    (modeling_df["FY"] >= OOT_START)
    &
    (modeling_df["FY"] <= OOT_END)
].copy()

# =============================================================================
# 5. SPLIT SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("TEMPORAL SPLIT")
print("=" * 80)

print(
    f"\nTRAIN       : 2012–{TRAIN_END}"
    f"\nVALIDATION  : {VALIDATION_START}–{VALIDATION_END}"
    f"\nOOT TEST    : {OOT_START}–{OOT_END}"
)

print("\nOBSERVATIONS")
print("-" * 80)

print(
    f"Training observations   : {len(train_df):,}"
)
print(
    f"Validation observations : {len(validation_df):,}"
)
print(
    f"OOT observations        : {len(oot_df):,}"
)

print("\nUNIQUE COMPANIES")
print("-" * 80)

print(
    f"Training companies   : {train_df['Company'].nunique():,}"
)
print(
    f"Validation companies : {validation_df['Company'].nunique():,}"
)
print(
    f"OOT companies        : {oot_df['Company'].nunique():,}"
)

# =============================================================================
# 6. TARGET DISTRIBUTION
# =============================================================================

label_order = [
    "Healthy",
    "Vulnerable",
    "Distressed"
]

print("\n" + "=" * 80)
print("TARGET DISTRIBUTION BY SAMPLE")
print("=" * 80)

def target_summary(data, name):

    counts = (
        data["DistressLabel"]
        .value_counts()
        .reindex(label_order, fill_value=0)
    )

    pct = (
        counts / len(data) * 100
    )

    out = pd.DataFrame({
        "Count": counts,
        "Percentage": pct.round(2)
    })

    print(f"\n{name}")
    print("-" * 60)
    print(out)

    return out


train_target_summary = target_summary(
    train_df,
    "TRAIN"
)

validation_target_summary = target_summary(
    validation_df,
    "VALIDATION"
)

oot_target_summary = target_summary(
    oot_df,
    "OOT 2022–2023"
)

# =============================================================================
# 7. YEAR-BY-YEAR TARGET DISTRIBUTION
# =============================================================================

print("\n" + "=" * 80)
print("TARGET DISTRIBUTION BY YEAR")
print("=" * 80)

year_target = pd.crosstab(
    modeling_df["FY"],
    modeling_df["DistressLabel"],
    normalize="index"
) * 100

year_target = (
    year_target
    .reindex(columns=label_order)
    .round(2)
)

print(year_target)

# =============================================================================
# 8. IDENTIFY FEATURE COLUMNS
# =============================================================================

excluded = {
    "Company",
    "FY",
    "DistressLabel",
    "Target"
}

feature_cols = [
    c for c in modeling_df.columns
    if c not in excluded
]

print("\n" + "=" * 80)
print("MODEL FEATURES")
print("=" * 80)

for i, c in enumerate(feature_cols, 1):
    print(f"{i:2d}. {c}")

print("\nTotal model features:", len(feature_cols))

# =============================================================================
# 9. EXPLICIT ICR SAFETY CHECK
# =============================================================================

print("\n" + "=" * 80)
print("LEAKAGE CHECK")
print("=" * 80)

for forbidden in [
    "ICR",
    "DistressLabel",
    "Target"
]:

    if forbidden in feature_cols:

        raise ValueError(
            f"CRITICAL LEAKAGE: {forbidden} is present in features."
        )

print("✓ ICR excluded.")
print("✓ DistressLabel excluded.")
print("✓ Target excluded.")

# =============================================================================
# 10. MISSINGNESS BY SPLIT
# =============================================================================

print("\n" + "=" * 80)
print("MISSINGNESS BY TEMPORAL SPLIT")
print("=" * 80)

missingness = []

for feature in feature_cols:

    missingness.append({
        "Feature": feature,
        "Train_Missing_%": (
            train_df[feature].isna().mean() * 100
        ),
        "Validation_Missing_%": (
            validation_df[feature].isna().mean() * 100
        ),
        "OOT_Missing_%": (
            oot_df[feature].isna().mean() * 100
        )
    })

missingness_df = pd.DataFrame(
    missingness
)

missingness_df["Max_Missing_%"] = (
    missingness_df[
        [
            "Train_Missing_%",
            "Validation_Missing_%",
            "OOT_Missing_%"
        ]
    ].max(axis=1)
)

missingness_df = (
    missingness_df
    .sort_values(
        "Max_Missing_%",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    missingness_df.to_string(
        index=False
    )
)

# =============================================================================
# 11. CHECK TRAINING-ONLY CONSTANT FEATURES
# =============================================================================

constant_features = []

for feature in feature_cols:

    if train_df[feature].nunique(
        dropna=True
    ) <= 1:

        constant_features.append(
            feature
        )

print("\n" + "=" * 80)
print("TRAINING-ONLY CONSTANT FEATURES")
print("=" * 80)

if constant_features:

    print(
        constant_features
    )

else:

    print(
        "None."
    )

# =============================================================================
# 12. IMPORTANT: MARKET-TO-BOOK REVIEW
# =============================================================================

print("\n" + "=" * 80)
print("MARKET-TO-BOOK REVIEW")
print("=" * 80)

if "MarketToBook" in modeling_df.columns:

    print(
        "MarketToBook missingness:"
    )

    print(
        f"Train       : "
        f"{train_df['MarketToBook'].isna().mean()*100:.2f}%"
    )

    print(
        f"Validation  : "
        f"{validation_df['MarketToBook'].isna().mean()*100:.2f}%"
    )

    print(
        f"OOT         : "
        f"{oot_df['MarketToBook'].isna().mean()*100:.2f}%"
    )

    print(
        "\nDecision: KEEP MarketToBook for now."
    )

    print(
        "Imputation will be fitted on TRAIN only."
    )

# =============================================================================
# 13. CHECK COMPANY OVERLAP ACROSS TIME
# =============================================================================

print("\n" + "=" * 80)
print("TEMPORAL COMPANY OVERLAP")
print("=" * 80)

train_companies = set(
    train_df["Company"].unique()
)

validation_companies = set(
    validation_df["Company"].unique()
)

oot_companies = set(
    oot_df["Company"].unique()
)

print(
    "Train ∩ Validation companies:",
    len(
        train_companies &
        validation_companies
    )
)

print(
    "Train ∩ OOT companies:",
    len(
        train_companies &
        oot_companies
    )
)

print(
    "Validation ∩ OOT companies:",
    len(
        validation_companies &
        oot_companies
    )
)

# =============================================================================
# 14. CLASS WEIGHT REQUIREMENT
# =============================================================================

print("\n" + "=" * 80)
print("CLASS IMBALANCE")
print("=" * 80)

train_counts = (
    train_df["Target"]
    .value_counts()
    .sort_index()
)

print(
    train_counts
)

class_weights = (
    len(train_df)
    /
    (
        len(train_counts)
        * train_counts
    )
)

class_weight_dict = (
    class_weights
    .to_dict()
)

print("\nSuggested balanced class weights:")
print(class_weight_dict)

# =============================================================================
# 15. SAVE SPLITS
# =============================================================================

TRAIN_PATH = (
    r"./data"
    r"\distress_train.csv"
)

VALIDATION_PATH = (
    r"./data"
    r"\distress_validation.csv"
)

OOT_PATH = (
    r"./data"
    r"\distress_oot.csv"
)

train_df.to_csv(
    TRAIN_PATH,
    index=False
)

validation_df.to_csv(
    VALIDATION_PATH,
    index=False
)

oot_df.to_csv(
    OOT_PATH,
    index=False
)

print("\n" + "=" * 80)
print("FILES SAVED")
print("=" * 80)

print(TRAIN_PATH)
print(VALIDATION_PATH)
print(OOT_PATH)

# =============================================================================
# 16. CREATE GLOBAL OBJECTS FOR NEXT STEPS
# =============================================================================

X_train = train_df[feature_cols].copy()
y_train = train_df["Target"].copy()

X_validation = validation_df[feature_cols].copy()
y_validation = validation_df["Target"].copy()

X_oot = oot_df[feature_cols].copy()
y_oot = oot_df["Target"].copy()

print("\n" + "=" * 80)
print("STEP 4 COMPLETE")
print("=" * 80)

print(
    "\n✓ Chronological split created."
)

print(
    "✓ 2012–2019 training set created."
)

print(
    "✓ 2020–2021 validation set created."
)

print(
    "✓ 2022–2023 true OOT test set created."
)

print(
    "✓ ICR leakage remains excluded."
)

print(
    "✓ MarketToBook retained for proper train-only imputation."
)

print(
    "\nNext: baseline multinomial logistic regression."
)

In [ ]:
# =============================================================================
# STEP 5 FIX — HANDLE INFINITY VALUES BEFORE IMPUTATION
# =============================================================================

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    log_loss,
    f1_score
)

print("=" * 80)
print("STEP 5 — BASELINE MULTINOMIAL LOGISTIC REGRESSION")
print("INFINITY-SAFE VERSION")
print("=" * 80)


# =============================================================================
# 1. LOAD DATA
# =============================================================================

train_df = pd.read_csv(
    r"./data/distress_train.csv"
)

validation_df = pd.read_csv(
    r"./data/distress_validation.csv"
)

oot_df = pd.read_csv(
    r"./data/distress_oot.csv"
)


# =============================================================================
# 2. FEATURE LIST
# =============================================================================

target_col = "Target"

feature_cols = [
    "Leverage_trend",
    "SalesGrowth_3yr_trend",
    "InstitutionalHolding_pct",
    "Industry_Median_Deviation",
    "DebtEquityRatio",
    "CashConversionCycle",
    "InventoryDays",
    "CreditorDays",
    "ROCE_3yr_trend",
    "SalesGrowth",
    "PromoterControlConcentration",
    "Accruals_to_Assets",
    "CFO_to_NetIncome",
    "CFO_to_TL",
    "DebtorDays",
    "CashBurnRunway",
    "TangibleAssetRatio",
    "AssetTurnover",
    "CashToAssets",
    "CurrentRatio",
    "FirmAge",
    "ROCE",
    "LogTotalAssets",
    "EquityRaised_t",
    "PromoterPledgeRatio",
    "PledgeDisclosed",
    "EquityRaised_dummy"
]


# =============================================================================
# 3. PREPARE DATA
# =============================================================================

X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()

X_validation = validation_df[feature_cols].copy()
y_validation = validation_df[target_col].copy()

X_oot = oot_df[feature_cols].copy()
y_oot = oot_df[target_col].copy()


# =============================================================================
# 4. CONVERT INFINITY TO MISSING VALUES
# =============================================================================

print("\n" + "=" * 80)
print("INFINITY AUDIT")
print("=" * 80)

for name, df in [
    ("Train", X_train),
    ("Validation", X_validation),
    ("OOT", X_oot)
]:

    inf_count = np.isinf(df.to_numpy(dtype=float)).sum()

    print(f"{name} infinity values: {inf_count}")


X_train = X_train.replace(
    [np.inf, -np.inf],
    np.nan
)

X_validation = X_validation.replace(
    [np.inf, -np.inf],
    np.nan
)

X_oot = X_oot.replace(
    [np.inf, -np.inf],
    np.nan
)

print("\n✓ ±Infinity converted to NaN.")
print("✓ Missing values will be handled by training-only median imputation.")


# =============================================================================
# 5. PREPROCESSING
# =============================================================================

preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# =============================================================================
# 6. CLASS WEIGHTS
# =============================================================================

class_weights = {
    0: 0.527594070695553,
    1: 1.7143386439422008,
    2: 1.9183250414593698
}


# =============================================================================
# 7. LOGISTIC REGRESSION MODEL
# =============================================================================

logit_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                multi_class="multinomial",
                class_weight=class_weights,
                max_iter=3000,
                solver="lbfgs",
                random_state=42
            )
        )
    ]
)


# =============================================================================
# 8. FIT
# =============================================================================

print("\n" + "=" * 80)
print("FITTING MODEL")
print("=" * 80)

logit_model.fit(
    X_train,
    y_train
)

print("✓ Model fitted successfully.")


# =============================================================================
# 9. PREDICTIONS
# =============================================================================

train_pred = logit_model.predict(X_train)
validation_pred = logit_model.predict(X_validation)
oot_pred = logit_model.predict(X_oot)

train_prob = logit_model.predict_proba(X_train)
validation_prob = logit_model.predict_proba(X_validation)
oot_prob = logit_model.predict_proba(X_oot)

print("✓ Train predictions generated.")
print("✓ Validation predictions generated.")
print("✓ OOT predictions generated.")


# 10. EVALUATION FUNCTION

def evaluate_model(
    y_true,
    y_pred,
    y_prob,
    sample_name
):

    return {
        "Sample": sample_name,
        "Observations": len(y_true),
        "Accuracy": accuracy_score(
            y_true,
            y_pred
        ),
        "Balanced_Accuracy": balanced_accuracy_score(
            y_true,
            y_pred
        ),
        "Macro_F1": f1_score(
            y_true,
            y_pred,
            average="macro"
        ),
        "Weighted_F1": f1_score(
            y_true,
            y_pred,
            average="weighted"
        ),
        "Log_Loss": log_loss(
            y_true,
            y_prob,
            labels=[0, 1, 2]
        ),
        "ROC_AUC_OVR": roc_auc_score(
            y_true,
            y_prob,
            multi_class="ovr",
            average="macro"
        )
    }


# =============================================================================
# 11. PERFORMANCE
# =============================================================================

train_results = evaluate_model(
    y_train,
    train_pred,
    train_prob,
    "Train 2012-2019"
)

validation_results = evaluate_model(
    y_validation,
    validation_pred,
    validation_prob,
    "Validation 2020-2021"
)

oot_results = evaluate_model(
    y_oot,
    oot_pred,
    oot_prob,
    "OOT 2022-2023"
)

baseline_results = pd.DataFrame(
    [
        train_results,
        validation_results,
        oot_results
    ]
)

print("\n" + "=" * 80)
print("BASELINE MODEL PERFORMANCE")
print("=" * 80)

print(
    baseline_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# =============================================================================
# 12. CLASSIFICATION REPORTS
# =============================================================================

class_names = [
    "Healthy",
    "Vulnerable",
    "Distressed"
]

for sample_name, y_true, y_pred in [
    ("TRAIN", y_train, train_pred),
    ("VALIDATION", y_validation, validation_pred),
    ("OOT", y_oot, oot_pred)
]:

    print("\n" + "=" * 80)
    print(f"{sample_name} CLASSIFICATION REPORT")
    print("=" * 80)

    print(
        classification_report(
            y_true,
            y_pred,
            labels=[0, 1, 2],
            target_names=class_names,
            digits=4
        )
    )


# =============================================================================
# 13. CONFUSION MATRICES
# =============================================================================

for sample_name, y_true, y_pred in [
    ("TRAIN", y_train, train_pred),
    ("VALIDATION", y_validation, validation_pred),
    ("OOT", y_oot, oot_pred)
]:

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1, 2]
    )

    print("\n" + "=" * 80)
    print(f"{sample_name} CONFUSION MATRIX")
    print("=" * 80)

    print(
        pd.DataFrame(
            cm,
            index=[
                "Actual Healthy",
                "Actual Vulnerable",
                "Actual Distressed"
            ],
            columns=[
                "Pred Healthy",
                "Pred Vulnerable",
                "Pred Distressed"
            ]
        )
    )


# =============================================================================
# 14. OOT CLASS-WISE PERFORMANCE
# =============================================================================

oot_report = classification_report(
    y_oot,
    oot_pred,
    labels=[0, 1, 2],
    target_names=class_names,
    output_dict=True
)

oot_class_performance = pd.DataFrame(
    {
        "Class": class_names,
        "Precision": [
            oot_report["Healthy"]["precision"],
            oot_report["Vulnerable"]["precision"],
            oot_report["Distressed"]["precision"]
        ],
        "Recall": [
            oot_report["Healthy"]["recall"],
            oot_report["Vulnerable"]["recall"],
            oot_report["Distressed"]["recall"]
        ],
        "F1": [
            oot_report["Healthy"]["f1-score"],
            oot_report["Vulnerable"]["f1-score"],
            oot_report["Distressed"]["f1-score"]
        ],
        "Support": [
            oot_report["Healthy"]["support"],
            oot_report["Vulnerable"]["support"],
            oot_report["Distressed"]["support"]
        ]
    }
)

print("\n" + "=" * 80)
print("OOT CLASS-WISE PERFORMANCE")
print("=" * 80)

print(
    oot_class_performance.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# =============================================================================
# 15. SAVE RESULTS
# =============================================================================

baseline_results.to_csv(
    r"./data/distress_baseline_results.csv",
    index=False
)

oot_class_performance.to_csv(
    r"./data/distress_baseline_oot_class_performance.csv",
    index=False
)


# =============================================================================
# 16. FINAL STATUS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 5 COMPLETE")
print("=" * 80)

print("✓ Infinity-safe preprocessing applied.")
print("✓ Median imputation fitted only on training data.")
print("✓ Standardization fitted only on training data.")
print("✓ Multinomial logistic baseline fitted.")
print("✓ Train performance calculated.")
print("✓ Validation performance calculated.")
print("✓ 2022–2023 OOT performance calculated.")
print("✓ Confusion matrices generated.")
print("✓ OOT class-level performance generated.")

print("\nSaved:")
print(
    r"./data/distress_baseline_results.csv"
)

print(
    r"./data/distress_baseline_oot_class_performance.csv"
)

print("\nDO NOT CHANGE THE MODEL YET.")
print("Inspect the baseline results before proceeding.")

In [ ]:
# =============================================================================
# FINANCIAL DISTRESS PROJECT — STEP 6
# NONLINEAR MODEL BENCHMARK — FIXED DATA LOADING
# =============================================================================

import os
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier


print("=" * 80)
print("FINANCIAL DISTRESS PROJECT — STEP 6")
print("NONLINEAR MODEL BENCHMARK")
print("=" * 80)


# =============================================================================
# 1. LOAD DATA
# =============================================================================

print("\n" + "=" * 80)
print("LOADING TEMPORAL DATASETS")
print("=" * 80)

train_path = r"./data/distress_train.csv"
validation_path = r"./data/distress_validation.csv"
oot_path = r"./data/distress_oot.csv"

if not os.path.exists(train_path):
    raise FileNotFoundError(
        f"Training file not found:\n{train_path}"
    )

if not os.path.exists(validation_path):
    raise FileNotFoundError(
        f"Validation file not found:\n{validation_path}"
    )

if not os.path.exists(oot_path):
    raise FileNotFoundError(
        f"OOT file not found:\n{oot_path}"
    )


train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(validation_path)
oot_df = pd.read_csv(oot_path)

print("✓ Training dataset loaded.")
print("✓ Validation dataset loaded.")
print("✓ OOT dataset loaded.")

print("\nTrain      :", train_df.shape)
print("Validation :", validation_df.shape)
print("OOT        :", oot_df.shape)


# =============================================================================
# 2. IDENTIFY FEATURE SET
# =============================================================================

print("\n" + "=" * 80)
print("IDENTIFYING MODEL FEATURES")
print("=" * 80)

if "feature_cols" not in globals():

    feature_cols = [
        "Leverage_trend",
        "SalesGrowth_3yr_trend",
        "InstitutionalHolding_pct",
        "Industry_Median_Deviation",
        "DebtEquityRatio",
        "CashConversionCycle",
        "InventoryDays",
        "CreditorDays",
        "ROCE_3yr_trend",
        "SalesGrowth",
        "PromoterControlConcentration",
        "Accruals_to_Assets",
        "CFO_to_NetIncome",
        "CFO_to_TL",
        "DebtorDays",
        "CashBurnRunway",
        "TangibleAssetRatio",
        "AssetTurnover",
        "CashToAssets",
        "CurrentRatio",
        "FirmAge",
        "ROCE",
        "LogTotalAssets",
        "EquityRaised_t",
        "PromoterPledgeRatio",
        "PledgeDisclosed",
        "EquityRaised_dummy"
    ]

    print("✓ Feature list reconstructed.")

else:

    print("✓ Existing feature_cols found.")


print("\nNumber of features:", len(feature_cols))


# =============================================================================
# 3. VERIFY REQUIRED COLUMNS
# =============================================================================

required_columns = feature_cols + ["Target"]

for name, df in [
    ("Train", train_df),
    ("Validation", validation_df),
    ("OOT", oot_df)
]:

    missing = [
        c for c in required_columns
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{name} is missing columns: {missing}"
        )

print("\n✓ All required columns are present.")


# =============================================================================
# 4. PREPARE X / y
# =============================================================================

X_train = train_df[feature_cols].copy()
X_validation = validation_df[feature_cols].copy()
X_oot = oot_df[feature_cols].copy()

y_train = train_df["Target"].astype(int)
y_validation = validation_df["Target"].astype(int)
y_oot = oot_df["Target"].astype(int)


# =============================================================================
# 5. INFINITY SAFETY
# =============================================================================

print("\n" + "=" * 80)
print("INFINITY AUDIT")
print("=" * 80)

for name, df in [
    ("Train", X_train),
    ("Validation", X_validation),
    ("OOT", X_oot)
]:

    infinity_count = np.isinf(
        df.to_numpy(dtype=float)
    ).sum()

    print(
        f"{name} infinity values: {infinity_count}"
    )

    df.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True
    )

print("\n✓ ±Infinity converted to NaN.")
print("✓ XGBoost will handle missing values natively.")


# =============================================================================
# 6. CLASS DISTRIBUTION
# =============================================================================

print("\n" + "=" * 80)
print("TRAINING CLASS DISTRIBUTION")
print("=" * 80)

print(
    y_train.value_counts()
    .sort_index()
)


# =============================================================================
# 7. CLASS WEIGHTS
# =============================================================================

class_counts = y_train.value_counts().sort_index()

n_total = len(y_train)
n_classes = len(class_counts)

class_weights = {
    int(cls): n_total / (
        n_classes * count
    )
    for cls, count in class_counts.items()
}

print("\nSuggested class weights:")
print(class_weights)

sample_weights = np.array([
    class_weights[int(y)]
    for y in y_train
])


# =============================================================================
# 8. FIT XGBOOST
# =============================================================================

print("\n" + "=" * 80)
print("TRAINING XGBOOST")
print("=" * 80)

xgb_model = XGBClassifier(

    objective="multi:softprob",
    num_class=3,

    n_estimators=500,
    learning_rate=0.03,
    max_depth=4,

    min_child_weight=5,

    subsample=0.85,
    colsample_bytree=0.85,

    reg_alpha=0.1,
    reg_lambda=2.0,

    random_state=42,

    eval_metric="mlogloss",

    tree_method="hist",
    n_jobs=-1
)


xgb_model.fit(
    X_train,
    y_train,
    sample_weight=sample_weights,

    eval_set=[
        (X_validation, y_validation)
    ],

    verbose=False
)

print("✓ XGBoost model fitted.")


# =============================================================================
# 9. GENERATE PREDICTIONS
# =============================================================================

print("\n" + "=" * 80)
print("GENERATING PREDICTIONS")
print("=" * 80)

train_prob = xgb_model.predict_proba(
    X_train
)

validation_prob = xgb_model.predict_proba(
    X_validation
)

oot_prob = xgb_model.predict_proba(
    X_oot
)

train_pred = np.argmax(
    train_prob,
    axis=1
)

validation_pred = np.argmax(
    validation_prob,
    axis=1
)

oot_pred = np.argmax(
    oot_prob,
    axis=1
)

print("✓ Train predictions generated.")
print("✓ Validation predictions generated.")
print("✓ OOT predictions generated.")


# =============================================================================
# 10. EVALUATION FUNCTION
# =============================================================================

def evaluate_multiclass(
    sample_name,
    y_true,
    y_pred,
    y_prob
):

    return {
        "Sample": sample_name,

        "Observations": len(y_true),

        "Accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "Balanced_Accuracy": balanced_accuracy_score(
            y_true,
            y_pred
        ),

        "Macro_F1": f1_score(
            y_true,
            y_pred,
            average="macro"
        ),

        "Weighted_F1": f1_score(
            y_true,
            y_pred,
            average="weighted"
        ),

        "Log_Loss": log_loss(
            y_true,
            y_prob
        ),

        "ROC_AUC_OVR": roc_auc_score(
            y_true,
            y_prob,
            multi_class="ovr",
            average="macro"
        )
    }


# =============================================================================
# 11. PERFORMANCE
# =============================================================================

xgb_results = pd.DataFrame([

    evaluate_multiclass(
        "Train 2012-2019",
        y_train,
        train_pred,
        train_prob
    ),

    evaluate_multiclass(
        "Validation 2020-2021",
        y_validation,
        validation_pred,
        validation_prob
    ),

    evaluate_multiclass(
        "OOT 2022-2023",
        y_oot,
        oot_pred,
        oot_prob
    )

])


print("\n" + "=" * 80)
print("XGBOOST MODEL PERFORMANCE")
print("=" * 80)

print(
    xgb_results.round(4).to_string(
        index=False
    )
)


# =============================================================================
# 12. OOT CLASSIFICATION REPORT
# =============================================================================

print("\n" + "=" * 80)
print("XGBOOST OOT CLASSIFICATION REPORT")
print("=" * 80)

print(
    classification_report(
        y_oot,
        oot_pred,
        target_names=[
            "Healthy",
            "Vulnerable",
            "Distressed"
        ],
        digits=4
    )
)


# =============================================================================
# 13. OOT CONFUSION MATRIX
# =============================================================================

print("\n" + "=" * 80)
print("XGBOOST OOT CONFUSION MATRIX")
print("=" * 80)

cm_oot = confusion_matrix(
    y_oot,
    oot_pred
)

cm_oot_df = pd.DataFrame(

    cm_oot,

    index=[
        "Actual Healthy",
        "Actual Vulnerable",
        "Actual Distressed"
    ],

    columns=[
        "Pred Healthy",
        "Pred Vulnerable",
        "Pred Distressed"
    ]
)

print(
    cm_oot_df.to_string()
)


# =============================================================================
# 14. CLASS-WISE OOT PERFORMANCE
# =============================================================================

report_oot = classification_report(
    y_oot,
    oot_pred,
    target_names=[
        "Healthy",
        "Vulnerable",
        "Distressed"
    ],
    output_dict=True
)


xgb_oot_class_performance = pd.DataFrame({

    "Class": [
        "Healthy",
        "Vulnerable",
        "Distressed"
    ],

    "Precision": [
        report_oot["Healthy"]["precision"],
        report_oot["Vulnerable"]["precision"],
        report_oot["Distressed"]["precision"]
    ],

    "Recall": [
        report_oot["Healthy"]["recall"],
        report_oot["Vulnerable"]["recall"],
        report_oot["Distressed"]["recall"]
    ],

    "F1": [
        report_oot["Healthy"]["f1-score"],
        report_oot["Vulnerable"]["f1-score"],
        report_oot["Distressed"]["f1-score"]
    ],

    "Support": [
        report_oot["Healthy"]["support"],
        report_oot["Vulnerable"]["support"],
        report_oot["Distressed"]["support"]
    ]

})


print("\n" + "=" * 80)
print("XGBOOST OOT CLASS-WISE PERFORMANCE")
print("=" * 80)

print(
    xgb_oot_class_performance.round(4)
    .to_string(index=False)
)


# =============================================================================
# 15. FEATURE IMPORTANCE
# =============================================================================

xgb_feature_importance = pd.DataFrame({

    "Feature": feature_cols,

    "Importance": (
        xgb_model.feature_importances_
    )

}).sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)


print("\n" + "=" * 80)
print("TOP 15 XGBOOST FEATURES")
print("=" * 80)

print(
    xgb_feature_importance
    .head(15)
    .round(6)
    .to_string(index=False)
)


# =============================================================================
# 16. SAVE RESULTS
# =============================================================================

results_path = (
    r"./data/distress_xgb_results.csv"
)

class_path = (
    r"./data/distress_xgb_oot_class_performance.csv"
)

importance_path = (
    r"./data/distress_xgb_feature_importance.csv"
)


xgb_results.to_csv(
    results_path,
    index=False
)

xgb_oot_class_performance.to_csv(
    class_path,
    index=False
)

xgb_feature_importance.to_csv(
    importance_path,
    index=False
)


# =============================================================================
# 17. FINAL
# =============================================================================

print("\n" + "=" * 80)
print("STEP 6 COMPLETE")
print("=" * 80)

print("✓ XGBoost fitted.")
print("✓ Validation performance calculated.")
print("✓ OOT performance calculated.")
print("✓ OOT class performance calculated.")
print("✓ Feature importance calculated.")

print("\nSaved:")
print(results_path)
print(class_path)
print(importance_path)

print("\nDO NOT CHANGE THE MODEL YET.")
print("Next: compare XGBoost against the logistic baseline.")

print("=" * 80)

In [ ]:
# =============================================================================
# STEP 7B — ICR PROXY CORRELATION CHECK
# USE ORIGINAL DATASET BECAUSE MODELING DATASET EXCLUDES ICR
# =============================================================================

print("\n" + "=" * 80)
print("ICR-PROXY CORRELATION CHECK")
print("=" * 80)

# -------------------------------------------------------------------------
# 1. LOAD ORIGINAL DATASET
# -------------------------------------------------------------------------

original_path = r"./data/distress_panel_FINAL.csv"

original_df = pd.read_csv(original_path)

print("Original dataset shape:", original_df.shape)

# -------------------------------------------------------------------------
# 2. CHECK ICR
# -------------------------------------------------------------------------

if "ICR" not in original_df.columns:
    raise ValueError(
        "ICR is not present in the original dataset."
    )

print("✓ ICR found in original dataset.")

# -------------------------------------------------------------------------
# 3. KEEP MODEL FEATURES THAT ALSO EXIST IN ORIGINAL DATA
# -------------------------------------------------------------------------

numeric_features = [
    c for c in feature_cols
    if c in original_df.columns
]

print(
    "\nModel features available in original dataset:",
    len(numeric_features)
)

# -------------------------------------------------------------------------
# 4. BUILD CORRELATION DATA
# -------------------------------------------------------------------------

corr_data = original_df[
    ["ICR"] + numeric_features
].copy()

corr_data = corr_data.replace(
    [np.inf, -np.inf],
    np.nan
)

# -------------------------------------------------------------------------
# 5. CORRELATION WITH ICR
# -------------------------------------------------------------------------

icr_corr = (
    corr_data
    .corr(numeric_only=True)["ICR"]
    .drop("ICR")
    .abs()
    .sort_values(ascending=False)
)

print("\n" + "=" * 80)
print("TOP FEATURES MOST STRONGLY CORRELATED WITH ICR")
print("=" * 80)

print(
    icr_corr
    .head(15)
    .round(4)
    .to_string()
)

# -------------------------------------------------------------------------
# 6. TOP XGBOOST FEATURES
# -------------------------------------------------------------------------

importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": xgb_model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

top_features = importance_df.head(15)["Feature"].tolist()

proxy_check = pd.DataFrame({
    "Feature": top_features,
    "XGB_Importance": [
        importance_df.loc[
            importance_df["Feature"] == f,
            "Importance"
        ].iloc[0]
        for f in top_features
    ],
    "Abs_ICR_Correlation": [
        icr_corr.get(f, np.nan)
        for f in top_features
    ]
})

print("\n" + "=" * 80)
print("TOP XGBOOST FEATURES VS ICR CORRELATION")
print("=" * 80)

print(
    proxy_check
    .round(4)
    .to_string(index=False)
)

# -------------------------------------------------------------------------
# 7. SAVE
# -------------------------------------------------------------------------

proxy_check.to_csv(
    r"./data/distress_icr_proxy_check.csv",
    index=False
)

importance_df.to_csv(
    r"./data/distress_xgb_importance_final.csv",
    index=False
)

comparison.to_csv(
    r"./data/distress_model_comparison.csv",
    index=False
)

# -------------------------------------------------------------------------
# 8. FINAL STATUS
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 7 COMPLETE")
print("=" * 80)

print("✓ Model comparison completed.")
print("✓ XGBoost feature importance reviewed.")
print("✓ ICR correlations calculated from ORIGINAL dataset.")
print("✓ No ICR included in the model.")
print("✓ Results saved.")

print("\nNext: interpret the ICR-proxy results before changing the model.")

## 5. XGBoost robustness and interpretability checks

These experiments document model behavior after leakage controls and provide robustness evidence. They are not separate final claims unless explicitly identified as such.
